# Assignment #6: Modern Data Structures
#### Natalie Masetti

### Preparation

In [1]:
# loading packages
import pandas as pd
import numpy as np
import os
import requests
from bs4 import BeautifulSoup
from urllib.request import urlretrieve
import math

# working direction
os.getcwd()

os.chdir(r"C:/Users/natal/OneDrive/Documents/GitHub/QMSS-GR5072-Spring2026-HWs/homework-6-main/Submissions/MASETTI-NATALIE") # working folder
path = r"C:/Users/natal/OneDrive/Documents/GitHub/QMSS-GR5072-Spring2026-HWs/homework-6-main/Submissions/MASETTI-NATALIE"

### Question #1
[10 pts] Web-scrape all table data from the following webpage and build a pd.DataFrame object. Limit your final table to include columns for "Package", "Item", "Title", "Rows", and "Cols". Print the dimensions and first five rows of the table.

In [2]:
html_string = 'https://vincentarelbundock.github.io/Rdatasets/datasets.html' # set html string to website

dfs = pd.read_html(html_string) # read html in
dfs_df = pd.DataFrame(dfs[0]) # make dataframe
dfs_q1 = dfs_df.loc[:, ['Package', 'Item', 'Title', 'Rows', 'Cols']] # select important cols
print(f'Dimensions:', dfs_q1.shape) # print dimensions
print(f'\n', dfs_q1.head()) # print head

Dimensions: (3500, 5)

   Package           Item                                              Title  \
0     AER        Affairs                   Fair's Extramarital Affairs Data   
1     AER   ArgentinaCPI                  Consumer Price Index in Argentina   
2     AER      BankWages                                         Bank Wages   
3     AER  BenderlyZwick  Benderly and Zwick Data: Inflation, Growth and...   
4     AER      BondYield                                    Bond Yield Data   

    Rows  Cols  
0  601.0   9.0  
1   80.0   2.0  
2  474.0   4.0  
3   31.0   5.0  
4   60.0   2.0  


### Question #2

[10 pts] Web-scrape the full links to every CSV file listed in the CSV column of the web-page. Add a new column to your data frame that includes these links. Name the column "csv_links". Print the first five rows of the table.

In [3]:
URL = "https://vincentarelbundock.github.io/Rdatasets/datasets.html"
page = requests.get(URL)
soup = BeautifulSoup(page.content, "html.parser") 

In [4]:
links = []
for link in soup.find_all('a'):
    links.append(link.get('href')) # from datacamp

In [5]:
links_df = pd.DataFrame(links) # make a dataframe of links list
links_df = links_df.rename(columns = {0:'link'}) # rename 
links_df = links_df.loc[::2] # there were copies of each link, took every other
links_df = links_df.reset_index(drop = True) 
linklist = links_df['link'] # set link list
dfs_q1['csv_links'] = linklist # rename link column

In [6]:
print(dfs_q1.head())

  Package           Item                                              Title  \
0     AER        Affairs                   Fair's Extramarital Affairs Data   
1     AER   ArgentinaCPI                  Consumer Price Index in Argentina   
2     AER      BankWages                                         Bank Wages   
3     AER  BenderlyZwick  Benderly and Zwick Data: Inflation, Growth and...   
4     AER      BondYield                                    Bond Yield Data   

    Rows  Cols                                          csv_links  
0  601.0   9.0  https://vincentarelbundock.github.io/Rdatasets...  
1   80.0   2.0  https://vincentarelbundock.github.io/Rdatasets...  
2  474.0   4.0  https://vincentarelbundock.github.io/Rdatasets...  
3   31.0   5.0  https://vincentarelbundock.github.io/Rdatasets...  
4   60.0   2.0  https://vincentarelbundock.github.io/Rdatasets...  


### Question #3
[5 pts] Search the "Title" column to return the row of data with the title "Violent Crime Rates by US State".

In [7]:
print(dfs_q1[dfs_q1['Title'] == 'Violent Crime Rates by US State'])

            Package          Item                            Title  Rows  \
838   crimedatasets  USArrests_df  Violent Crime Rates by US State  50.0   
1068       datasets     USArrests  Violent Crime Rates by US State  50.0   
3294     usdatasets  USArrests_df  Violent Crime Rates by US State  50.0   

      Cols                                          csv_links  
838    4.0  https://vincentarelbundock.github.io/Rdatasets...  
1068   4.0  https://vincentarelbundock.github.io/Rdatasets...  
3294   4.0  https://vincentarelbundock.github.io/Rdatasets...  


### Question #4
[10 pts] Import the csv file for the dataset identified in #3 using the full link listed in the "csv_links" column for this dataset. Create a new variable called "violent_crime" that adds together data for all columns in the dataset that contain violent crime data (i.e.-add data from assault, murder, and rape columns together in new column called "violent_crime").

In [8]:
data_link = dfs_q1['csv_links'].loc[838] # I chose one of the 3 options there were
urlretrieve(data_link, 'USArrests_df.csv')

('USArrests_df.csv', <http.client.HTTPMessage at 0x27a883ffbf0>)

In [9]:
crime = pd.read_csv("USArrests_df.csv") 

In [10]:
crime['violent_crime'] = crime['Murder'] + crime['Assault'] + crime['Rape']

### Question #5
[10 pts] Merge all the data from the states.csv data set (found in the data folder of this HW6 directory) with your dataset to your Violent Crime Rates data frame. Print the first five lines of your new dataset.

In [11]:
states = pd.read_csv('states.csv')

In [12]:
full = states.merge(crime, left_on = 'state.name', right_on = 'rownames', how = 'inner')

In [13]:
print(full.head())

   state.name state.abb  state.area  state.division state.name.1  \
0     Alabama        AL       51609               4      Alabama   
1      Alaska        AK      589757               9       Alaska   
2     Arizona        AZ      113909               8      Arizona   
3    Arkansas        AR       53104               5     Arkansas   
4  California        CA      158693               9   California   

   state.region  Population  Income  Illiteracy  Life.Exp  Murder_x  HS.Grad  \
0             2        3615    3624         2.1     69.05      15.1     41.3   
1             4         365    6315         1.5     69.31      11.3     66.7   
2             4        2212    4530         1.8     70.55       7.8     58.1   
3             2        2110    3378         1.9     70.66      10.1     39.9   
4             4       21198    5114         1.1     71.71      10.3     62.6   

   Frost    Area    rownames  Murder_y  Assault  UrbanPop  Rape  violent_crime  
0     20   50708     Alabama 

### Question #6:
[5 pts] Calculate the average for each numeric column in the dataset.

In [14]:
print(full.mean(numeric_only = True))

state.area        72367.9800
state.division        5.2000
state.region          2.5800
Population         4246.4200
Income             4435.8000
Illiteracy            1.1700
Life.Exp             70.8786
Murder_x              7.3780
HS.Grad              53.1080
Frost               104.4600
Area              70735.8800
Murder_y              7.7880
Assault             170.7600
UrbanPop             65.5400
Rape                 21.2320
violent_crime       199.7800
dtype: float64


### Question #7
[10 pts] Group the data by region and then calculate the average for each numeric column in the dataset per region. Which region had the highest population (data is from the late 1970s)? Which region had the most violent crime?

In [15]:
grouped = full.groupby(['state.region']).mean(numeric_only = True)

In [16]:
print(grouped.sort_values('Population', ascending = False))

                 state.area  state.division   Population       Income  \
state.region                                                            
1              18817.000000        1.333333  5495.111111  4570.222222   
3              63794.166667        6.583333  4803.000000  4611.083333   
2              56222.250000        3.750000  4208.125000  4011.937500   
4             137227.692308        8.384615  2915.307692  4702.615385   

              Illiteracy   Life.Exp   Murder_x    HS.Grad       Frost  \
state.region                                                            
1               1.000000  71.264444   4.722222  53.966667  132.777778   
3               0.700000  71.766667   5.275000  54.516667  138.833333   
2               1.737500  69.706250  10.581250  44.343750   64.625000   
4               1.023077  71.234615   7.215385  62.000000  102.153846   

                    Area   Murder_y     Assault   UrbanPop       Rape  \
state.region                                     

In [17]:
print(grouped.sort_values('violent_crime', ascending = False))

                 state.area  state.division   Population       Income  \
state.region                                                            
2              56222.250000        3.750000  4208.125000  4011.937500   
4             137227.692308        8.384615  2915.307692  4702.615385   
1              18817.000000        1.333333  5495.111111  4570.222222   
3              63794.166667        6.583333  4803.000000  4611.083333   

              Illiteracy   Life.Exp   Murder_x    HS.Grad       Frost  \
state.region                                                            
2               1.737500  69.706250  10.581250  44.343750   64.625000   
4               1.023077  71.234615   7.215385  62.000000  102.153846   
1               1.000000  71.264444   4.722222  53.966667  132.777778   
3               0.700000  71.766667   5.275000  54.516667  138.833333   

                    Area   Murder_y     Assault   UrbanPop       Rape  \
state.region                                     

Region 1, which looks to be the northeast, had the highest population (5495), whereas region 2 had the highest number of violent crimes (252).

### Question #8
[5 pts] What SQL statement would you write to return two columns denoting income and Illiteracy in your state data?

SELECT Income, Illiteracy
FROM full;

### Question #10
[10 pts] What SQL statement would you write to return two columns denoting income and Illiteracy in your state data and sort the data from the highest to lowest income values and limit the data to incomes at or higher than 5000?

SELECT Income, Illiteracy
FROM full
WHERE Income >= 5000
ORDER BY Income DESC;

### Question #11
[10 pts] Create a new data frame that includes two columns from your state data denoting state names and violent crimes. Spread the state names to 50 unique columns with a single row that includes the violent crime data per state. Print the first five columns of the new dataset.

In [18]:
full_new = full.loc[:,['state.name', 'violent_crime']]

In [19]:
full_new= full.pivot(columns="state.name", values="violent_crime") 

In [20]:
full_new = full_new.transform(np.sort) # stackoverflow

In [21]:
full_new = full_new.loc[0,:]

In [22]:
full_new = pd.DataFrame(full_new)

In [23]:
print(full_new.head())

                0
state.name       
Alabama     270.4
Alaska      317.5
Arizona     333.1
Arkansas    218.3
California  325.6


### Question #11
[10 pts] Take the dataset from question 10 and use a function to return a dictionary with each key denoting a state and each value indicating the square root of the value for violent crimes.

In [24]:
def dict_builder(df, value_col):
    df[value_col] = [math.sqrt(row) for row in df[value_col]]
    return df.to_dict()

In [25]:
dict_q11 = dict_builder(full_new, 0)
dict_q11 = dict_q11[0]

In [26]:
print(dict_q11)

{'Alabama': 16.443843832875572, 'Alaska': 17.81852968120546, 'Arizona': 18.251027368342857, 'Arkansas': 14.774978849392646, 'California': 18.04438970982394, 'Colorado': 15.830350596243914, 'Connecticut': 11.153474794878948, 'Delaware': 16.11521020650987, 'Florida': 19.552493447128423, 'Georgia': 15.943650773897426, 'Hawaii': 8.455767262643882, 'Idaho': 11.696153213770756, 'Illinois': 16.834488409215172, 'Indiana': 11.882760622010357, 'Iowa': 8.336666000266533, 'Kansas': 11.789826122551595, 'Kentucky': 11.61895003862225, 'Louisiana': 16.929264603047585, 'Maine': 9.638464608017191, 'Maryland': 18.414668066516974, 'Massachusetts': 13.026895255585654, 'Michigan': 17.383900597967074, 'Minnesota': 9.465727652959385, 'Mississippi': 17.093858546273278, 'Missouri': 14.669696656713798, 'Montana': 11.46298390472568, 'Nebraska': 11.081516141756055, 'Nevada': 17.61249556422939, 'New Hampshire': 8.282511696339462, 'New Jersey': 13.608820668963201, 'New Mexico': 18.12456896039186, 'New York': 17.0645

### Question #12
[5 pts] Subset the list you created in question 12 to extract values for Texas and New York.

In [28]:
ny_tx = ['New York', 'Texas']

for key, value in dict_q11.items():
    if key in ny_tx:
        print(key + " " + str(value)) # datacamp

New York 17.064583206161235
Texas 15.466091943344964


# Bibliography
- Bowne-Anderson, H., Vankrunkelsven, V., & Schouwenaars, F. (n.d.). *Loop over dictionary* \[Data Camp lecture\]. In Bowne-Anderson, H., Vankrunkelsven, V., & Schouwenaars, F., *Intermediate Python.* DataCamp. https://campus.datacamp.com/courses/intermediate-python/loops?ex=11

- Bowne-Anderson, H., Castro, F. (n.d.). *Turning a webpage into data using BeautifulSoup: getting the hyperlinks* \[Data Camp lecture\]. In Bowne-Anderson, H., Castro, F., *Intermediate Importing Data in Python.* DataCamp. https://campus.datacamp.com/courses/intermediate-importing-data-in-python/importing-data-from-the-internet-1?ex=12

- TheBamf. (2019, December 20). *Answer to: Sort all columns of a dataframe* \[Source Code\]. StackOverflow. https://stackoverflow.com/questions/41507040/sort-all-columns-of-a-dataframe

- Hicks, R. (2026, February 11). *week_03.ipynb* \[Source code\]. Jupyter Notebook

- Hicks, R. (2026, February 19). *week_04.ipynb* \[Source code\]. Jupyter Notebook
 
- Hicks, R. (2026, February 19). *week_05.ipynb* \[Source code\]. Jupyter Notebook
   
- Hicks, R. (2026, March 9). *week_07.ipynb* \[Source code\]. Jupyter Notebook
     
- Hicks, R. (2026, April 16). *week_12.ipynb* \[Source code\]. Jupyter Notebook